# Pairwise developmental changes (zero lag)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from notebook_setup import init
FINAL_DIR, PDF_DIR = init()
import plot_style

from analysis_config import BASE_PATH, FOCUS_AUS, ALPHA
from dyad_analysis_utils import (
    load_openface_data,
    load_prepared_meetings,
    organize_sessions_by_period,
    compute_meeting_fisher_matrices,
    collect_fisher_values_per_au_pair,
    run_paired_comparisons,
    compute_cohens_d_matrix,
    holm_correct_matrix,
    plot_comparison_pvalue_heatmaps,
    plot_comparison_cohens_d_heatmaps,
)

alpha = ALPHA
base_path = BASE_PATH
if not base_path.exists():
    raise FileNotFoundError(f"Expected directory not found: {base_path.resolve()}")

meeting_groups, input_paths = load_openface_data(base_path)
print(f"Found {len(input_paths)} session folders, {len(meeting_groups)} meetings")
print(f"PDF export dir: {PDF_DIR}")

In [ ]:
df_infant_arr, df_adult_arr, AUs = load_prepared_meetings(meeting_groups)
print(f"Loaded {len(df_infant_arr)} meetings")


In [ ]:
period_aggregated_data, all_periods_data = organize_sessions_by_period(
    df_infant_arr, df_adult_arr, AUs
)
for p in (1, 2, 3):
    print(f"Period {p}: {len(period_aggregated_data[p])} sessions")


In [ ]:
print(f"Using FOCUS_AUS ({len(FOCUS_AUS)}): {FOCUS_AUS}")


In [ ]:
meeting_fisher_matrices = {
    meeting_id: compute_meeting_fisher_matrices(
        df_infant_arr[meeting_id], df_adult_arr[meeting_id], FOCUS_AUS
    )
    for meeting_id in df_infant_arr
}
print(f"Computed Fisher Z matrices for {len(meeting_fisher_matrices)} meetings")


In [ ]:
au_pair_fisher_values = collect_fisher_values_per_au_pair(meeting_fisher_matrices, FOCUS_AUS)

In [ ]:
comparisons = [(1, 2), (1, 3), (2, 3)]
test_name = "Wilcoxon signed-rank test"

pvalue_matrices = run_paired_comparisons(au_pair_fisher_values, FOCUS_AUS, comparisons)

for comp in comparisons:
    n_valid = (~pvalue_matrices[comp].isna()).sum().sum()
    n_sig = (pvalue_matrices[comp] < alpha).sum().sum()
    label = f"{plot_style.month_label(comp[0])} vs {plot_style.month_label(comp[1])}"
    print(f"  {label}: {n_sig}/{n_valid} significant (p < {alpha})")


In [ ]:
plot_comparison_pvalue_heatmaps(
    pvalue_matrices,
    comparisons,
    FOCUS_AUS,
    alpha=alpha,
    test_label=f"{test_name}; Fisher Z",
    save_path=PDF_DIR / "algo1_pvalues_heatmaps.pdf",
)


In [ ]:
cohens_d_matrices = {
    comp: compute_cohens_d_matrix(au_pair_fisher_values, comp[0], comp[1], FOCUS_AUS)
    for comp in comparisons
}

plot_comparison_cohens_d_heatmaps(
    cohens_d_matrices,
    pvalue_matrices,
    comparisons,
    FOCUS_AUS,
    test_label=f"{test_name}; Fisher Z",
    save_path=PDF_DIR / "algo1_cohens_d_heatmaps.pdf",
)

print("\nCohen's d: negative = higher correlation in earlier month; * p≤.05, ** p≤.01")


In [ ]:
pvalue_matrices_holm = {
    comp: holm_correct_matrix(pvalue_matrices[comp], alpha=alpha) for comp in comparisons
}
for comp in comparisons:
    p_raw, p_holm = pvalue_matrices[comp], pvalue_matrices_holm[comp]
    n_tests = int(p_raw.notna().values.sum())
    n_raw = int((p_raw.values < alpha).sum())
    n_holm = int((p_holm.values < alpha).sum())
    label = f"{plot_style.month_label(comp[0])} vs {plot_style.month_label(comp[1])}"
    print(f"{label}: {n_raw}/{n_tests} uncorrected, {n_holm}/{n_tests} after Holm")
    for au_infant in FOCUS_AUS:
        for au_adult in FOCUS_AUS:
            if p_holm.loc[au_infant, au_adult] < alpha:
                print(
                    f"    infant {au_infant} x maternal {au_adult}: "
                    f"p = {p_raw.loc[au_infant, au_adult]:.5f}, "
                    f"p_Holm = {p_holm.loc[au_infant, au_adult]:.4f}, "
                    f"d = {cohens_d_matrices[comp].loc[au_infant, au_adult]:.2f}"
                )
